# Комп'ютерна графіка — тренажер «Три проекції»

## Єдина просторова модель → три взаємопов'язані проекції

Точка A(x, y, z) автоматично має три проекції:

- **A₁(X, Z)** — фронтальна;
- **A₂(X, Y)** — горизонтальна;
- **A₃(Y, Z)** — профільна.

Те саме для B і C. Зміна однієї координати змінює тільки ті проекції, у яких ця координата присутня.

**Запустіть одну наступну code-комірку.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox

# ============================================================
# КОМП'ЮТЕРНА ГРАФІКА
# Інтерактивний тренажер системи трьох проекцій
# ============================================================

COLORS = {"A": "tab:red", "B": "tab:blue", "C": "tab:green"}


def make_points(Ax, Ay, Az, Bx, By, Bz, Cx, Cy, Cz):
    return np.array([
        [Ax, Ay, Az],
        [Bx, By, Bz],
        [Cx, Cy, Cz]
    ], dtype=float)


def classify_plane(P):
    normal = np.cross(P[1] - P[0], P[2] - P[0])

    if np.linalg.norm(normal) < 1e-9:
        return "⚠️ Площина не визначена: точки A, B, C колінеарні"

    if np.allclose(P[:, 2], P[0, 2]):
        return "Горизонтальна площина  •  Z = const"

    if np.allclose(P[:, 1], P[0, 1]):
        return "Фронтальна площина  •  Y = const"

    if np.allclose(P[:, 0], P[0, 0]):
        return "Профільна площина  •  X = const"

    return "Площина загального положення"


def common_limits(P):
    low = P.min(axis=0)
    high = P.max(axis=0)

    span = np.maximum(high - low, 4)
    margin = 0.15 * span

    return low - margin, high + margin


def draw_point(ax, x, y, name, color):
    ax.scatter(x, y, s=70, color=color, zorder=5)
    ax.annotate(
        name,
        (x, y),
        xytext=(7, 7),
        textcoords="offset points",
        fontsize=12,
        fontweight="bold"
    )


def draw_front(ax, P, limits, show_lines):
    low, high = limits

    ax.set_xlim(low[0], high[0])
    ax.set_ylim(low[2], high[2])
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("X")
    ax.set_ylabel("Z")
    ax.set_title("ФРОНТАЛЬНА  XZ", fontweight="bold")
    ax.grid(True, alpha=0.25)

    Q = P[:, [0, 2]]
    ax.plot(Q[[0, 1, 2, 0], 0], Q[[0, 1, 2, 0], 1], color="black")

    for i, name in enumerate(["A₁", "B₁", "C₁"]):
        draw_point(ax, Q[i, 0], Q[i, 1], name, COLORS["ABC"[i]])

        if show_lines:
            ax.axvline(Q[i, 0], color=COLORS["ABC"[i]], ls="--", alpha=0.18)


def draw_horizontal(ax, P, limits, show_lines):
    low, high = limits

    ax.set_xlim(low[0], high[0])
    ax.set_ylim(low[1], high[1])
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_title("ГОРИЗОНТАЛЬНА  XY", fontweight="bold")
    ax.grid(True, alpha=0.25)

    Q = P[:, [0, 1]]
    ax.plot(Q[[0, 1, 2, 0], 0], Q[[0, 1, 2, 0], 1], color="black")

    for i, name in enumerate(["A₂", "B₂", "C₂"]):
        draw_point(ax, Q[i, 0], Q[i, 1], name, COLORS["ABC"[i]])

        if show_lines:
            ax.axvline(Q[i, 0], color=COLORS["ABC"[i]], ls="--", alpha=0.18)


def draw_profile(ax, P, limits, show_lines):
    low, high = limits

    ax.set_xlim(low[1], high[1])
    ax.set_ylim(low[2], high[2])
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Y")
    ax.set_ylabel("Z")
    ax.set_title("ПРОФІЛЬНА  YZ", fontweight="bold")
    ax.grid(True, alpha=0.25)

    Q = P[:, [1, 2]]
    ax.plot(Q[[0, 1, 2, 0], 0], Q[[0, 1, 2, 0], 1], color="black")

    for i, name in enumerate(["A₃", "B₃", "C₃"]):
        draw_point(ax, Q[i, 0], Q[i, 1], name, COLORS["ABC"[i]])

        if show_lines:
            ax.axvline(Q[i, 0], color=COLORS["ABC"[i]], ls="--", alpha=0.18)


def draw_projection_connections(ax_front, ax_horizontal, ax_profile, P, limits):
    low, high = limits

    # Спільний простір координат. Лінії показують,
    # які проекції належать одній просторовій точці.
    for i, name in enumerate(["A", "B", "C"]):
        color = COLORS[name]

        # У межах кожного поля залишаємо тонкі напрямні.
        ax_front.axvline(P[i, 0], color=color, ls=":", alpha=0.15)
        ax_horizontal.axvline(P[i, 0], color=color, ls=":", alpha=0.15)

        ax_horizontal.axhline(P[i, 1], color=color, ls=":", alpha=0.15)
        ax_profile.axvline(P[i, 1], color=color, ls=":", alpha=0.15)

        ax_front.axhline(P[i, 2], color=color, ls=":", alpha=0.15)
        ax_profile.axhline(P[i, 2], color=color, ls=":", alpha=0.15)


def draw_3d(ax, P, limits):
    low, high = limits

    ax.set_xlim(low[0], high[0])
    ax.set_ylim(low[1], high[1])
    ax.set_zlim(low[2], high[2])

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title("ДОПОМІЖНИЙ 3D-ВИГЛЯД", fontweight="bold")

    contour = [0, 1, 2, 0]

    ax.plot(
        P[contour, 0],
        P[contour, 1],
        P[contour, 2],
        color="black",
        linewidth=1.5
    )

    normal = np.cross(P[1] - P[0], P[2] - P[0])

    if np.linalg.norm(normal) > 1e-9:
        ax.plot_trisurf(
            P[:, 0],
            P[:, 1],
            P[:, 2],
            alpha=0.12
        )
    else:
        ax.text2D(
            0.05,
            0.92,
            "Площина не визначена",
            transform=ax.transAxes,
            fontweight="bold"
        )

    for i, name in enumerate(["A", "B", "C"]):
        ax.scatter(*P[i], s=70, color=COLORS[name])
        ax.text(*P[i], "  " + name, fontweight="bold")

    ax.view_init(elev=24, azim=-58)


def trainer(
    Ax=2, Ay=2, Az=2,
    Bx=8, By=3, Bz=6,
    Cx=5, Cy=8, Cz=4,
    show_lines=True
):
    P = make_points(
        Ax, Ay, Az,
        Bx, By, Bz,
        Cx, Cy, Cz
    )

    limits = common_limits(P)

    fig = plt.figure(figsize=(16, 9))

    ax_front = fig.add_subplot(221)
    ax_horizontal = fig.add_subplot(223)
    ax_profile = fig.add_subplot(222)
    ax_3d = fig.add_subplot(224, projection="3d")

    draw_front(ax_front, P, limits, show_lines)
    draw_horizontal(ax_horizontal, P, limits, show_lines)
    draw_profile(ax_profile, P, limits, show_lines)

    if show_lines:
        draw_projection_connections(
            ax_front,
            ax_horizontal,
            ax_profile,
            P,
            limits
        )

    draw_3d(ax_3d, P, limits)

    fig.suptitle(
        classify_plane(P),
        fontsize=17,
        fontweight="bold"
    )

    plt.tight_layout()
    plt.show()


interact(
    trainer,

    Ax=FloatSlider(
        min=0, max=10, step=1, value=2,
        description="A: X",
        continuous_update=False
    ),
    Ay=FloatSlider(
        min=0, max=10, step=1, value=2,
        description="A: Y",
        continuous_update=False
    ),
    Az=FloatSlider(
        min=0, max=10, step=1, value=2,
        description="A: Z",
        continuous_update=False
    ),

    Bx=FloatSlider(
        min=0, max=10, step=1, value=8,
        description="B: X",
        continuous_update=False
    ),
    By=FloatSlider(
        min=0, max=10, step=1, value=3,
        description="B: Y",
        continuous_update=False
    ),
    Bz=FloatSlider(
        min=0, max=10, step=1, value=6,
        description="B: Z",
        continuous_update=False
    ),

    Cx=FloatSlider(
        min=0, max=10, step=1, value=5,
        description="C: X",
        continuous_update=False
    ),
    Cy=FloatSlider(
        min=0, max=10, step=1, value=8,
        description="C: Y",
        continuous_update=False
    ),
    Cz=FloatSlider(
        min=0, max=10, step=1, value=4,
        description="C: Z",
        continuous_update=False
    ),

    show_lines=Checkbox(
        value=True,
        description="Показувати зв'язки"
    )
)


## Експеримент

1. Змініть тільки **A: X**. Спостерігайте A₁ та A₂. A₃ не повинен змінюватися.
2. Поверніть A: X назад і змініть тільки **A: Y**. Тепер змінюються A₂ та A₃.
3. Змініть тільки **A: Z**. Тепер змінюються A₁ та A₃.

Це дозволяє побачити головний принцип:

**X → фронтальна + горизонтальна**  
**Y → горизонтальна + профільна**  
**Z → фронтальна + профільна**